# 01 - Prepare Crop Dataset (Manifest, Dedup-safe Split)

Stage 1: Dataset preparation for the CROP IDENTIFICATION model.

What this does:
1. Walks your raw dataset folder (Crop -> Environment -> Disease -> images)
2. Builds a manifest.csv recording every image with crop/disease/environment
   (disease + environment are kept as metadata for later per-crop disease
   models -- this stage only trains on the crop label)
3. Deduplicates near-identical images using perceptual hashing (important
   for the Groundnut "Created Dataset" folder, which looks derived from
   the Uncontrolled Environment folder)
4. Does a stratified train/val/test split by (crop, environment) so every
   split contains both controlled and uncontrolled images
5. Copies files into an ImageFolder-compatible layout:
      dataset_split/train/<crop>/xxx.jpg
      dataset_split/val/<crop>/xxx.jpg
      dataset_split/test/<crop>/xxx.jpg

Install deps first:
    pip install imagehash pillow pandas scikit-learn tqdm --break-system-packages

## Imports & Configuration

In [9]:
import os
import shutil
from pathlib import Path
from collections import defaultdict

import pandas as pd
from PIL import Image
import imagehash
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# ---- CONFIG -----------------------------------------------------------
RAW_ROOT = Path(r"Z:\Projects\Smart-Farming\Datasets\final_combined_datasets")
OUT_MANIFEST = Path(r"Z:\Projects\Smart-Farming\models\csvs\manifest.csv")
OUT_SPLIT_DIR = Path(r"Z:\Projects\Smart-Farming\Datasets\dataset_split")
VALID_EXT = {".jpg", ".jpeg", ".png", ".bmp"}

# Map raw top-level folder names -> canonical crop label.
# Add entries here if your folder names differ.
CROP_FOLDER_MAP = {
    "CLCE": ("Cotton", "closed"),
    "CLUE": ("Cotton", "uncontrolled"),
    "GLCDfGLUE": ("Groundnut", "derived"),
    "GLFCE": ("Groundnut", "closed"),
    "GLUE": ("Groundnut", "uncontrolled"),
    "PBCE": ("Pepper Bell", "closed"),
    "PBUE": ("Pepper Bell", "uncontrolled"),
    "PLUE": ("Potato", "uncontrolled"),
    "PLCE": ("Potato", "closed"),
    "TLCE": ("Tomato", "uncontrolled"),
    "TLUE": ("Tomato", "closed"),
}

## `build_manifest`

In [10]:
def build_manifest():
    rows = []
    for top_folder, (crop, environment) in CROP_FOLDER_MAP.items():
        crop_dir = RAW_ROOT / top_folder
        if not crop_dir.exists():
            print(f"WARNING: folder not found, skipping: {crop_dir}")
            continue
        for disease_dir in crop_dir.iterdir():
            if not disease_dir.is_dir():
                continue
            disease_raw = disease_dir.name
            for img_path in disease_dir.rglob("*"):
                if img_path.suffix.lower() in VALID_EXT:
                    rows.append({
                        "filepath": str(img_path),
                        "crop": crop,
                        "disease_raw": disease_raw,
                        "environment": environment,
                    })
    df = pd.DataFrame(rows)
    print(f"Found {len(df)} images across {df['crop'].nunique()} crops")
    print(df.groupby("crop").size())
    return df

## `drop_corrupted`

In [11]:
def drop_corrupted(df):
    good_rows = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Checking images"):
        try:
            with Image.open(row["filepath"]) as im:
                im.verify()
            good_rows.append(row)
        except Exception:
            print(f"Corrupted, dropping: {row['filepath']}")
    return pd.DataFrame(good_rows)

## `dedupe`

Flags near-IDENTICAL images (same crop) using perceptual hashing.

In [12]:
def dedupe(df, hash_size=8):
    """
    Flags near-IDENTICAL images (same crop) using perceptual hashing.
    Keeps the first occurrence, drops the rest.

    Note: this does NOT catch the Groundnut "Created Dataset" case, since
    those are cropped sub-regions of the Uncontrolled Environment photos,
    not identical/near-identical full images -- phash on a crop vs. the
    full photo won't match. That leakage risk is handled separately in
    split_dataset() below by locking derived-crop rows into the train
    split only.
    """
    seen = defaultdict(set)  # crop -> set of hashes already kept
    keep_rows = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Hashing / deduping"):
        try:
            with Image.open(row["filepath"]) as im:
                h = imagehash.phash(im, hash_size=hash_size)
        except Exception:
            continue
        crop = row["crop"]
        # Treat as duplicate if within a small hamming distance of any kept hash
        is_dup = any((h - existing) <= 4 for existing in seen[crop])
        if not is_dup:
            seen[crop].add(h)
            row = row.copy()
            row["phash"] = str(h)
            keep_rows.append(row)
    deduped = pd.DataFrame(keep_rows)
    print(f"Kept {len(deduped)} / {len(df)} images after dedup")
    print(deduped.groupby("crop").size())
    return deduped

## `split_dataset`

IMPORTANT: rows with environment == "derived" (the Groundnut "Created

In [13]:
def split_dataset(df, train_size=0.7, val_size=0.15, test_size=0.15, seed=42):
    """
    IMPORTANT: rows with environment == "derived" (the Groundnut "Created
    Dataset" crops) are locked into TRAIN only. They are cropped from the
    same source photos as "uncontrolled" Groundnut images, so letting them
    land in val/test risks a leaked/near-identical scene being scored as
    "correct" -- inflating validation accuracy without reflecting real
    performance. This also has a nice side effect: your val/test accuracy
    for Groundnut now only reflects whole-plant photos run through your
    own OpenCV leaf-detection crop, matching production conditions.
    """
    assert abs(train_size + val_size + test_size - 1.0) < 1e-6

    locked_train = df[df["environment"] == "derived"]
    splittable = df[df["environment"] != "derived"]

    # Stratify by crop + environment together so both environments show up
    # in every split, not just train.
    strat_key = splittable["crop"] + "_" + splittable["environment"]

    train_df, temp_df = train_test_split(
        splittable, train_size=train_size, stratify=strat_key, random_state=seed
    )
    train_df = pd.concat([train_df, locked_train], ignore_index=True)
    remaining_key = temp_df["crop"] + "_" + temp_df["environment"]
    relative_val = val_size / (val_size + test_size)
    val_df, test_df = train_test_split(
        temp_df, train_size=relative_val, stratify=remaining_key, random_state=seed
    )
    print(f"Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}")
    return train_df, val_df, test_df

## `materialize_split`

In [14]:
def materialize_split(split_name, split_df):
    for _, row in tqdm(split_df.iterrows(), total=len(split_df), desc=f"Copying {split_name}"):
        dest_dir = OUT_SPLIT_DIR / split_name / row["crop"]
        dest_dir.mkdir(parents=True, exist_ok=True)
        src = Path(row["filepath"])
        dest = dest_dir / f"{src.stem}_{abs(hash(str(src)))}{src.suffix}"
        shutil.copy2(src, dest)

## `main`

In [15]:
def main():
    df = build_manifest()
    df = drop_corrupted(df)
    df = dedupe(df)
    df.to_csv(OUT_MANIFEST, index=False)
    print(f"Manifest saved to {OUT_MANIFEST} (kept for disease-stage training later)")

    train_df, val_df, test_df = split_dataset(df)
    materialize_split("train", train_df)
    materialize_split("val", val_df)
    materialize_split("test", test_df)
    print("Done. dataset_split/{train,val,test}/<crop>/ is ready for training.")

## Run

In [16]:
main()

Found 34548 images across 5 crops
crop
Cotton         2658
Groundnut      7343
Pepper Bell    9574
Potato         7834
Tomato         7139
dtype: int64


Hashing / deduping: 100%|██████████| 34548/34548 [11:18<00:00, 50.91it/s] 


Kept 30404 / 34548 images after dedup
crop
Cotton         1924
Groundnut      6639
Pepper Bell    7522
Potato         7675
Tomato         6644
dtype: int64
Manifest saved to Z:\Projects\Smart-Farming\models\csvs\manifest.csv (kept for disease-stage training later)
Train: 21885  Val: 4259  Test: 4260


Copying test: 100%|██████████| 4260/4260 [02:21<00:00, 30.08it/s]

Done. dataset_split/{train,val,test}/<crop>/ is ready for training.
